# P133 — Los modelos de IA colapsan al entrenarse con datos generados recursivamente

## 1. Título y paper

**Paper:** *AI models collapse when trained on recursively generated data*  
**Autoría:** Ilia Shumailov, Zakhar Shumaylov, Yiren Zhao, Nicolas Papernot, Ross Anderson, Yarin Gal  
**Año y venue:** 2024 · Nature, 631, 755–759  
**Nivel:** L2 · **Motor:** `colapso_de_modelo`  
**Ficha completa:** [`P133_colapso_de_modelo`](../../papers/foundational/P133_colapso_de_modelo/README.md)

**Hito:** Demuestra que entrenar generación tras generación con datos sintéticos estrecha la distribución de forma irreversible, sin que ningún modelo cometa error alguno.

- [doi:10.1038/s41586-024-07566-y](https://doi.org/10.1038/s41586-024-07566-y)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La web se está llenando de texto e imágenes generadas. Los corpus futuros se recogerán de ahí, y nadie sabía qué le ocurre a un modelo entrenado sobre lo que generó la generación anterior.
2. Ejecutar una implementación mínima de la propuesta: Formalizar y medir el fenómeno en modelos de lenguaje, autocodificadores variacionales y mezclas de gaussianas: el error de muestreo acumulado basta para que las colas desaparezcan primero y la distribución converja a algo degenerado.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P131
- P61


## 4. Intuición

Treinta generaciones entrenando cada una con lo que generó la anterior. Ningún modelo comete un error: todos ajustan bien los datos que reciben. Y aun así la distribución se estrecha hasta perder dos tercios de su variedad.


## 5. Concepto mínimo

```text
gen 0 : datos reales           σ = 0,793
gen 30: solo generado          σ = 0,252      ← 31,8 % de la variedad

El mecanismo es SOLO el error de muestreo, acumulado
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('colapso_de_modelo', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánta variedad queda tras 30 generaciones?
2. ¿Qué le pasa al rango de los datos?
3. ¿Ayuda conservar datos reales?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('colapso_de_modelo', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('colapso_de_modelo', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Queda el **31,8 %** de la desviación original. El rango se estrecha **3,2×**. Y hay un segundo efecto: la distribución también **deriva** — la media pasa de −0,087 a −1,447. Conservar un **25 %** de datos reales por generación lo frena por completo: σ final **0,806** frente a **0,252**.


## 10. Comentario pedagógico

Esa deriva es lo que hace el fenómeno indetectable desde dentro. Cada generación ajusta perfectamente los datos que recibe y no tiene con qué comparar. Solo mirando la distribución original —que ya nadie tiene— se vería el desplazamiento. Por eso los corpus anteriores a la IA generativa se han vuelto un recurso con valor.


## 11. Error o anti-patrón deliberado

Anti-patrón: usar datos sintéticos sin registrar su procedencia.


In [ ]:
print('El problema no es entrenar con datos sinteticos: es no SABER que lo estas haciendo.')
print('Sin procedencia no puedes medir tu proporcion ni conservar datos reales a proposito.')
print('De ahi que P131 y P133 sean el mismo problema visto desde los dos lados.')

## 12. Corrección

El colapso, generación a generación:


In [ ]:
r = run_paper_lab('colapso_de_modelo', seed=3)['result']
for g in r['historia'][::5]:
    print(g)
print('conservado:', r['fraccion_conservada'])
print('con datos reales:', r['efecto_de_conservar_datos_reales'])

## 13. Desafío guiado

Explica por qué el colapso no exige ningún fallo del modelo, y qué lo distingue de un sobreajuste.


In [ ]:
r = run_paper_lab('colapso_de_modelo', seed=3)['result']
show(r)

## 14. Desafío autónomo

Estima qué fracción de tus datos de entrenamiento podría ser generada. Si no puedes estimarla, eso ya es el hallazgo: escribe qué registro te haría falta.


## 15. Evidencia de aprendizaje

Guarda la estimación —o la constatación de que no puedes hacerla— y el registro que necesitarías.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P133_colapso_de_modelo/README.md) · evaluación formal: [`assessments/papers/P133_colapso_de_modelo.md`](../../assessments/papers/P133_colapso_de_modelo.md)


## 16. Cierre

Cierra la ruta de medios. Lo siguiente son los agentes en operación: permisos, presupuestos y coordinación.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
